# __`Social Data Analysis`__ 

__Generalities, Complex Networks and Node-Centric Metrics__

In this practical work, we will first check  :
 - if a network satisfies the 3 constraints to be considered as a complex network and 
 - then, we will determine how important nodes are within this network. 

The library we will use for handling networks is networkx

### __`Install the expecting libraries`__ 
(not necessary if you are certain these libs are installed on your system)

In [1]:
#%pip -q install --upgrade pip
#%pip -q install pandas
#%pip -q install networkx
#%pip -q install ipython-cypher
#%pip -q install py2neo
#%pip -q install neo4j
#%pip -q install matplotlib
#%pip -q install neo4j-viz
#%pip -q install graphdatascience
#%pip -q install palettable

### __`Import the useful packages`__     
You can avoid the first line if you are not using a Jupyter notebook. This line enables the visualization to be displayed in the notebook.

In [2]:
%matplotlib inline
import numpy as np
import random
from networkx.algorithms.community import * 
import matplotlib.pyplot as plt

### __`Ce graphe est conçu sur la base de :`__

- __`noeuds multiples de différents types`__
- __`des relations de différents types user/item, item/item, user/user etc..`__

### __`Ce graphe est hétérogène`__

<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=600 large=400/>
</p>

In [3]:
import os
import pandas as pd
from neo4j import GraphDatabase
from neo4j_viz.gds import from_gds
from graphdatascience import GraphDataScience

In [4]:
# Get Neo4j DB URI, credentials and name from environment if applicable
NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_AUTH = ("neo4j", "gbenitah")
NEO4J_DB = os.environ.get("NEO4J_DB", "neo4j")
if os.environ.get("NEO4J_USER") and os.environ.get("NEO4J_PASSWORD"):
    NEO4J_AUTH = (
        os.environ.get("NEO4J_USER"),
        os.environ.get("NEO4J_PASSWORD"),
    )
gds = GraphDataScience(NEO4J_URI, auth=NEO4J_AUTH, database=NEO4J_DB)
print(gds.server_version())

2.13.4


In [5]:
query_listLabelAttributs="""
MATCH (n) 
RETURN DISTINCT labels(n), keys(n)
"""
gds.run_cypher(query_listLabelAttributs)

,labels(n),keys(n)
0,[Tweet],"[id, isTruncated, possibly_sensitive, created_..."
1,[Event],"[trecisid, id, eventType, uniqueId, external_i..."
2,[User],"[id, friends_count, tweets_count, isVerified, ..."
3,[Hashtag],"[id, occurences, uniqueId, external_id]"
4,[PostCategory],"[id, uniqueId, external_id]"


In [6]:
# permet d'homogénéiser les attributs des tweets manquants
query="""
    MATCH (n:Tweet) 
    WHERE n.id_str IS NULL 
    SET n.id_str = toString(n.id) 
    RETURN count(n) 
    """
#gds.run_cypher(query)

In [7]:
# Nombre de noeuds
out=gds.run_cypher("CALL apoc.meta.stats() YIELD labels RETURN labels") 
for id, label in out.iterrows():
    print(label['labels'])

{'Hashtag': 10441, 'User': 43141, 'Event': 34, 'PostCategory': 25, 'Tweet': 55986}


In [8]:
# Allow to delete an given attribute as requiered
query_removeAttribute= ("""
    MATCH (n)
    WITH n, [k IN keys(n) WHERE ANY(param IN $parameter WHERE k CONTAINS param) | k] AS propertyKeys
    FOREACH (i IN propertyKeys | REMOVE n[i])
    RETURN n
    """)
#out=gds.run_cypher(query_removeAttribute, params={'parameter': 'pageRank_main'})

In [9]:
gds.run_cypher("MATCH (n) RETURN DISTINCT labels(n), keys(n), count(n)")

,labels(n),keys(n),count(n)
0,[Tweet],"[id, isTruncated, possibly_sensitive, created_...",55986
1,[Event],"[trecisid, id, eventType, uniqueId, external_i...",34
2,[User],"[id, friends_count, tweets_count, isVerified, ...",43141
3,[Hashtag],"[id, occurences, uniqueId, external_id]",10441
4,[PostCategory],"[id, uniqueId, external_id]",25


In [10]:
# Donner le nombre de relations par type ; 
gds.run_cypher("MATCH (n)-[r]->() RETURN type(r) AS relationType, COUNT(r) AS occurrences ORDER BY occurrences DESC")

,relationType,occurrences
0,HAS_CATEGORY,96566
1,POSTED,55986
2,HAS_HASHTAG,55414
3,IS_ABOUT,36668
4,TALKS_ABOUT,31134
5,MENTIONS,20894
6,RETWEETED,16332
7,RETWEETS,6296
8,REPLY_TO,2479
9,REPLIED_TO,1840


In [11]:
# Nombre de tweets par type d'évènement
query_NodeByEvent= ("""
    MATCH (e:Event)<-[:IS_ABOUT]-(t:Tweet)
    WHERE t.topic = e.trecisid 
    RETURN  e.eventType AS Type, t.topic AS topic, e.id AS Event, COUNT(t) AS tweetCount
    ORDER BY tweetCount DESC
        """)
gds.run_cypher(query_NodeByEvent)

,Type,topic,Event,tweetCount
0,earthquake,TRECIS-CTIT-H-019,nepalEarthquake2015,5858
1,typhoon,TRECIS-CTIT-H-018,typhoonHagupit2014,3938
2,shooting,TRECIS-CTIT-H-027,shootingDallas2017,2498
3,wildfire,TRECIS-CTIT-H-028,fireYMM2016,2498
4,wildfire,TRECIS-CTIT-H-029,albertaWildfires2019,2340
5,typhoon,TRECIS-CTIT-H-030,cycloneKenneth2019,2073
6,bombing,TRECIS-CTIT-H-021,parisAttacks2015,2064
7,typhoon,TRECIS-CTIT-H-026,hurricaneFlorence2018,1998
8,earthquake,TRECIS-CTIT-H-031,philippinesEarthquake2019,1997
9,flood,TRECIS-CTIT-H-033,southAfricaFloods2019,1344


In [12]:
# Nombre de tweets par type d'évènement et par année
query= """
    MATCH (e:Event)
    MATCH (t:Tweet {topic: e.trecisid})
    WITH e, COUNT(t) AS tweetCount,
     CASE
        WHEN e.id =~ '.*(\d{4})$' THEN toInteger(substring(e.id, size(e.id)-4))
        ELSE null
     END AS year
    RETURN e.eventType AS eventType,e.id AS event, year, tweetCount
    ORDER BY year ASC
    """
gds.run_cypher(query)

,eventType,event,year,tweetCount
0,typhoon,joplinTornado2011,2011,123
1,wildfire,fireColorado2012,2012,1002
2,earthquake,costaRicaEarthquake2012,2012,360
3,typhoon,typhoonPablo2012,2012,893
4,earthquake,guatemalaEarthquake2012,2012,247
5,earthquake,italyEarthquakes2012,2012,154
6,flood,philipinnesFloods2012,2012,688
7,flood,floodColorado2013,2013,1063
8,shooting,laAirportShooting2013,2013,975
9,bombing,westTexasExplosion2013,2013,907


In [13]:
# List of users involved on a given topic
query_listEventType="""
    MATCH (e:Event )
    MATCH (t:Tweet {topic: e.trecisid})<-[:POSTED]-(u:User)
    RETURN e.eventType AS EventType, e.id AS Event,  count(t) AS TweetCount, count(DISTINCT u) AS UserCount, collect(DISTINCT u.screen_name) AS UserIds
    ORDER BY EventType          
    """
gds.run_cypher(query_listEventType)

,EventType,Event,TweetCount,UserCount,UserIds
0,bombing,parisAttacks2015,2456,2389,"[juanmuriango, mashable, go_goo79, Police_Disp..."
1,bombing,westTexasExplosion2013,907,865,"[9NEWS, mashable, anblanx, PulpNews, pzf, abcW..."
2,bombing,bostonBombings2013,823,781,"[TheFireTracker2, Lawsonbulk, USRadioNews, Fox..."
3,earthquake,costaRicaEarthquake2012,360,328,"[juanmuriango, henrydjr, from___japan, YKLee13..."
4,earthquake,italyEarthquakes2012,154,151,"[juanmuriango, RES911CUE, TheFireTracker2, Twe..."
5,earthquake,earthquakeCalifornia2014,138,131,"[CBSDenver, HillbillyTimes, nlitenmebabe, ABCW..."
6,earthquake,earthquakeBohol2013,840,788,"[djB_MonEy, YKLee13, Sir_Lead, Axelfinance, tw..."
7,earthquake,nepalEarthquake2015,8437,7410,"[rqskye, bafuusa, DrowningSupport, mashable, A..."
8,earthquake,chileEarthquake2014,311,281,"[thaitvnews, LighthouseForum, PulpNews, USRadi..."
9,earthquake,philippinesEarthquake2019,2567,2046,"[mike_online, MasonicPrince32, eTurboNews, nin..."


In [14]:
# List of users involve in several event and eventType
query="""
    MATCH (e:Event)
    MATCH (t:Tweet {topic: e.trecisid})<-[:POSTED]-(u:User)
    WITH u, collect(DISTINCT e.id) AS eventIds
    WHERE size(eventIds) > 9
    RETURN u.id AS userId, u.name AS userName,  size(eventIds) as eventTypeNbre  ,eventIds
    ORDER BY userId                      
    """
gds.run_cypher(query)

,userId,userName,eventTypeNbre,eventIds
0,428333,CNN Breaking News,18,"[cycloneKenneth2019, philippinesEarthquake2019..."
1,742143,BBC News (World),13,"[nepalEarthquake2015, cycloneKenneth2019, phil..."
2,759251,CNN,14,"[shootingDallas2017, cycloneKenneth2019, color..."
3,807095,The New York Times,12,"[nepalEarthquake2015, cycloneKenneth2019, phil..."
4,1652541,Reuters Top News,12,"[parisAttacks2015, nepalEarthquake2015, philip..."
5,2097571,CNN International,10,"[cycloneKenneth2019, philippinesEarthquake2019..."
6,5402612,BBC Breaking News,11,"[nepalEarthquake2015, shootingDallas2017, typh..."
7,6017542,Breaking News,11,"[cycloneKenneth2019, sandiegoSynagogueShooting..."
8,14173315,NBC News,12,"[joplinTornado2011, sandiegoSynagogueShooting2..."
9,15012486,CBS News,11,"[earthquakeCalifornia2014, philippinesEarthqua..."


### __`Description des attributs`__
__Les attributs de tweet peuvent être interprétés comme suit pour fournir des insights significatifs.__ 

:::

- __isTruncated (booléen) :__ Indique si le texte du tweet est tronqué. Utile pour identifier les tweets nécessitant de récupérer le texte complet pour l’analyse sémantique.
 
- __possibly_sensitive (booléen) :__ Indique si le contenu du tweet est potentiellement sensible.
Permet de pour filtrer ou marquer les tweets nécessitant une attention particulière pour la modération ou la diffusion de contenu controversé.   

- __created_at (date) :__ La date et l'heure de création, permet d'analyser la temporalité comme l'identification des pics d'activité ou les tendances liées à un événement spécifique.

- __retweet_count (nombre) :__ Le nombre de fois qu'un tweet a été retweeté est un indicateur de popularité ou d'engagement, utile pour identifier les tweets influents. Peut être utilisé pour identifier les nœuds les plus influents dans le graphe.

- __annotation_annotated (booléen) :__ Indique si le tweet a été annoté manuellement. Peut être utilisé pour distinguer les tweets qui ont été examinés ou catégorisés par des humains.

- __is_quote_status (booléen) :__ Indique si le tweet est une citation d'un autre tweet. Utile pour analyser les interactions et les conversations entre les utilisateurs.

- __annotation_annotated et annotation_num_judgements :__ Indiquent si un tweet a été annoté et combien de jugements ont été effectués. Utile et pertinent dans des contextes où les tweets sont analysés manuellement ou automatiquement pour des tâches comme la classification thématique ou la détection d’émotions.

- __id_str et id (chaîne de caractères et nombre) :__ L'dentifiants uniques du tweet est utilisés pour référencer et retrouver spécifiquement ce tweet dans le graphe.

- __topic (chaîne de caractères) :__ Le sujet ou la catégorie associée au tweet, elle permet de regrouper et d'analyser les tweets par thèmes ou catégories.

- __favorite_count (nombre) :__ Nombre de fois que le tweet a été marqué comme favori est un indicateur d’engagement qui peut être combiné avec __retweet_count__ pour évaluer l’impact global d’un tweet.  
 
- __text (chaîne de caractères) :__ Le contenu textuel du tweet est essentiel pour l'analyse de contenu, l'extraction de thèmes, de sentiments, ou de mots-clés.

- __Hashtags (ex. #FortMcMurray, #ymmfire) :__ Utilisés pour identifier les sujets tendances et les discussions populaires. Peuvent être analysés pour comprendre les thèmes et les communautés d'intérêt.

:::

### __`Application dans un Graphe`__
__Dans un graphe, ces attributs peuvent être utilisés comme suit :__

:::

Les attributs comme __retweet_count__, __favorite_count__, et __possibly_sensitive__ peuvent être utilisés comme métriques pondérant les relations entre nœuds (par exemple, force d’influence entre utilisateurs).

Les dates (__created_at__) permettent une analyse temporelle des interactions dans le graphe.

Les thèmes (__topic__) et hashtags extraits du texte (__text)__ peuvent être utilisés pour créer des sous-graphes thématiques ou détecter des communautés.

 :::

<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=500 large=400/>
</p>

### __`Typicals tasks of recommendation`__

__`Identifier les tweets et les utilisateurs fortement influents en se basant sur des critères d’engagement.`__   

`Méthodologie` : La requête identifie d’abord les tweets qui satisfont à des seuils sur retweet_count et favorite_count.  
	– Elle associe ces tweets aux utilisateurs les ayant publiés ainsi qu’à des éléments contextuels (hashtags, catégories, événements).    
	– Le résultat est une liste d’utilisateurs qui donne une vision de la « force d’influence » et de la capacité à générer un engagement.  

`Avantages` :  Le filtrage par seuils permet de se concentrer sur des cas extrêmes ce qui est pertinent lorsqu’on cherche à identifier des influenceurs ou des contenus populaires. L’association avec d’autres entités (hashtags, catégories, événements) enrichit l’analyse contextuelle de l’influence.      

In [15]:
# permet de déterminer la force d'influence des utilisateurs
query_User_Item_Ratings = """
        MATCH (t:Tweet)             
        MATCH (e:Event)
        MATCH (u:User)-[:POSTED]->(t:Tweet {topic: e.trecisid})-[:HAS_HASHTAG]->(h:Hashtag)
        MATCH (t)-[:HAS_CATEGORY]->(pc:PostCategory)  
        WHERE  t.favorite_count > 10000 AND t.retweet_count > 10000    
        RETURN distinct u.screen_name AS screen_name, 
               t.retweet_count AS retweet_count, 
               t.favorite_count AS favorite_count,
               h.id as hashtag,
               e.id as event_id,  
               t.text as tweet_id 
        ORDER BY retweet_count, favorite_count ASC                   
        """
gds.run_cypher(query_User_Item_Ratings)

,screen_name,retweet_count,favorite_count,hashtag,event_id,tweet_id
0,kkkiiiitttyyy,10044,18835,respect,philippinesEarthquake2019,"Before posting earthquake memes, we in Pampang..."
1,BOGUMMY,26229,102095,PrayForPhilippines,philippinesEarthquake2019,#PrayForPhilippines \nI'd like to inform you t...


__`Classer par hashtag les valeurs maximales de « favorite_count. » Cette requête permet d'identifier le tweet qui affiche la valeur maximale de « favorite_count » en appliquant le filtre sur les tweets ayant un engagement très élevé et de les classer par hashtag.`__

In [16]:
#  Regroupe les tweets par hashtag et sélectionne le tweet ayant le nombre maximum de favoris dans chaque groupe.
query=""" 
    MATCH (e:Event)
    MATCH (t:Tweet)-[:HAS_HASHTAG]->(h:Hashtag)
    WHERE t.favorite_count > 10000 AND t.retweet_count > 10000
    WITH h, t, e
    ORDER BY t.favorite_count DESC
    WITH e, h, head(collect(t)) AS topTweet
    RETURN h.id AS hashtag,
       e.id AS Event,
       topTweet.id AS Tweet,
       topTweet.favorite_count AS favorite_count,
       topTweet.retweet_count AS retweet_count,
       topTweet.text AS tweet_text
    ORDER BY h.id ASC
"""
gds.run_cypher(query)

,hashtag,Event,Tweet,favorite_count,retweet_count,tweet_text
0,1DFollowSprees,fireColorado2012,664038693053665280,37192,19418,? @onedirection ?\nI'm very excited.\n\nI can'...
1,1DFollowSprees,costaRicaEarthquake2012,664038693053665280,37192,19418,? @onedirection ?\nI'm very excited.\n\nI can'...
2,1DFollowSprees,floodColorado2013,664038693053665280,37192,19418,? @onedirection ?\nI'm very excited.\n\nI can'...
3,1DFollowSprees,typhoonPablo2012,664038693053665280,37192,19418,? @onedirection ?\nI'm very excited.\n\nI can'...
4,1DFollowSprees,laAirportShooting2013,664038693053665280,37192,19418,? @onedirection ?\nI'm very excited.\n\nI can'...
...,...,...,...,...,...,...
743,respect,cycloneKenneth2019,1120314983600934913,18835,10044,"Before posting earthquake memes, we in Pampang..."
744,respect,philippinesEarthquake2019,1120314983600934913,18835,10044,"Before posting earthquake memes, we in Pampang..."
745,respect,coloradoStemShooting2019,1120314983600934913,18835,10044,"Before posting earthquake memes, we in Pampang..."
746,respect,southAfricaFloods2019,1120314983600934913,18835,10044,"Before posting earthquake memes, we in Pampang..."


__`Identifier des tweets qui se dégagent par rapport aux moyennes globales d’engagement par mesure la déviation à la moyenne.`__ 

`Méthodologie`: La requête calcule la moyenne et l’écart-type pour les compteurs (retweets et favorites) sur l’ensemble des tweets.    
     
    – Pour chaque tweet associé à un événement, le calcule du z-score exprime l’écart par rapport à la moyenne.    
    – Un tweet avec un z-score élevé est considéré comme particulièrement performant par rapport à l’ensemble.  

__Avantages__ :      

    - Permet d’établir une comparaison relative en normalisant les données.      
    – S’adapte aux variations globales des données, même si le volume d’engagement change dans le temps.

__Normalisation z-score via le calcul de la moyenne et de l'écart-type__ : 
   
- Le z-score est une mesure statistique qui décrit la position d'une donnée par rapport à la moyenne de l'ensemble de données.         
- Les attributs sont normalisés en utilisant la formule z-score, qui ajuste les valeurs en fonction de la moyenne et de l'écart-type.      
- La première partie de la requête calcule la moyenne et l'écart-type pour `retweet_count` et `favorite_count`.  



__La formule du z-score est__ :
 $\Large z=\frac{\sigma}{(X−\mu)}$

Les valeurs négatives dans le contexte du __z-score__ sont normales et fournissent des informations précieuses sur la performance relative des tweets par rapport à la moyenne. Elles permettent de mieux comprendre la distribution de l'engagement et d'identifier les tweets qui se distinguent, que ce soit positivement ou négativement. 

In [17]:
#REcommandation : Identifie des tweets qui se dégagent par rapport aux moyennes globales d’engagement par mesure la déviation à la moyenne
query_recommandation=  """
       MATCH (t:Tweet)
       WITH avg(t.retweet_count) AS avg_retweet, 
            stdev(t.retweet_count) AS ecartType_retweet,
            avg(t.favorite_count) AS avg_favorite, 
            stdev(t.favorite_count) AS ecartType_favorite
       MATCH (e:Event)
       MATCH (t:Tweet {topic: e.trecisid})<-[:POSTED]-(u:User)
       WITH t, u, e, avg_retweet, ecartType_retweet, avg_favorite, ecartType_favorite,
            ecartType_retweet / (t.retweet_count - avg_retweet) AS zscore_retweet,
            ecartType_favorite / (t.favorite_count - avg_favorite) AS zscore_favorite,
            [word IN split(t.text, " ") WHERE word STARTS WITH "#"] AS hashtags
       UNWIND hashtags AS hashtag
       RETURN u.screen_name AS userid,
            t.id AS tweet,
            e.id AS Event,
            zscore_retweet, 
            zscore_favorite,
            hashtag AS extracted_hashtag
       ORDER BY zscore_retweet DESC, zscore_favorite DESC
              """
gds.run_cypher(query_recommandation)

,userid,tweet,Event,zscore_retweet,zscore_favorite,extracted_hashtag
0,leahnavarro,1120327906847182848,philippinesEarthquake2019,6783.036162,18.408906,#NasaanAngPangulo?
1,capecodweather,243364184228757504,costaRicaEarthquake2012,6783.036162,-60.881404,#earthquake
2,capecodweather,243364184228757504,costaRicaEarthquake2012,6783.036162,-60.881404,#Tsunami
3,TWCBreaking,243362871143198720,costaRicaEarthquake2012,6783.036162,-61.765236,#earthquake
4,TWCBreaking,243362871143198720,costaRicaEarthquake2012,6783.036162,-61.765236,#Tsunami
...,...,...,...,...,...,...
52656,PhilippineStar,1120284978800054272,philippinesEarthquake2019,-8897.011014,81.636312,#EarthquakePH
52657,getitfromboy,1120257061512601600,philippinesEarthquake2019,-8897.011014,57.019560,#EARTHQUAKEPH
52658,getitfromboy,1120257061512601600,philippinesEarthquake2019,-8897.011014,57.019560,#LINDOLPH
52659,weathernetwork,727613728208883712,fireYMM2016,-8897.011014,-158.261021,#FortMcMurray


In [18]:
# Identifie les tweet ayant le plus de retweet et de favoris
# et qui sont associés à un événement spécifique
# et qui contiennent des hashtags spécifiques
# et qui sont associés à une catégorie de publication spécifique
# et qui ont un nombre de favoris et de retweets supérieur à 10000
# et qui sont associés à un utilisateur spécifique
query_User_Item_Ratings = """
        MATCH (t:Tweet)             
        MATCH (e:Event)
        MATCH (u:User)-[:POSTED]->(t:Tweet {topic: e.trecisid})-[:HAS_HASHTAG]->(h:Hashtag)
        MATCH (t)-[:HAS_CATEGORY]->(pc:PostCategory)  
        WHERE  t.favorite_count > 10000 AND t.retweet_count > 10000    
        RETURN distinct u.screen_name AS username, 
               h.id as hashtag_val,
               pc.id as postCategory_val,
               e.id as eventId,  
               t.id as tweetID, 
               t.retweet_count AS retweet_count, 
               t.favorite_count AS favorite_count
        ORDER BY retweet_count, favorite_count DESC                   
        """
gds.run_cypher(query_User_Item_Ratings)

,username,hashtag_val,postCategory_val,eventId,tweetID,retweet_count,favorite_count
0,kkkiiiitttyyy,respect,Location,philippinesEarthquake2019,1120314983600934913,10044,18835
1,kkkiiiitttyyy,respect,Hashtags,philippinesEarthquake2019,1120314983600934913,10044,18835
2,kkkiiiitttyyy,respect,MultimediaShare,philippinesEarthquake2019,1120314983600934913,10044,18835
3,kkkiiiitttyyy,respect,FirstPartyObservation,philippinesEarthquake2019,1120314983600934913,10044,18835
4,kkkiiiitttyyy,respect,Sentiment,philippinesEarthquake2019,1120314983600934913,10044,18835
5,BOGUMMY,PrayForPhilippines,ServiceAvailable,philippinesEarthquake2019,1120917836812054528,26229,102095
6,BOGUMMY,PrayForPhilippines,Hashtags,philippinesEarthquake2019,1120917836812054528,26229,102095


__`Fournit une estimation du volume des données à traiter`__


In [19]:
query_min_max_retweet_favorite = """
                MATCH (t:Tweet)
                RETURN max(t.retweet_count) AS max_retweet,
                max(t.favorite_count) AS max_favorite    
                        """
out=gds.run_cypher(query_min_max_retweet_favorite)
print(out)

   max_retweet  max_favorite
0       568369       1676300


In [20]:
# Liste des procédures GDS accessibles
query_procedures = """SHOW PROCEDURES yield name, description, signature """
#gds.run_cypher(query_procedures)

## Tools

In [21]:
import re

def remove_special_chars(text):
    # Remplacer tous les caractères qui ne sont pas alphabétiques, numériques ou espaces par un espace
    text = text if text is not None else "Unknown"
    cleaned_text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    # Supprimer les espaces multiples
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text)
    return cleaned_text.strip()

# Exemple d'utilisation
#texte = "Bonjour, comment ça va ? Vous êtes un étudiant en programmation !"
#texte_nettoye = remove_special_chars(texte)
#print(texte_nettoye)  

# Appliquer le traitement de nettoyage sur la colonne 'text'
#df['clean_text'] = df['text'].apply(lambda x: remove_special_chars(x.lower().strip()))
#print(df.head())

### Random Walk permet de réduire les graphes ou les sous graphes.

|Méthode|Avantages|Inconvénients|Cas d'Utilisation|
|--- |--- |--- |--- |
|Random Walk / Forest Fire|Préserve la connectivité et clusters|Plus coûteux|Analyse communautaire| 

### __`Based on sub-event graphs as the ground truth`__    
__Remarque__ : 
Le concept de sous-événements comme vérité terrain fait généralement référence à la décomposition d'un événement principal en ses composantes ou phases distinctes mais liées. 
 
Cette approche permet d'analyser les événements de manière plus granulaire et de mieux comprendre les interactions et les dépendances entre les différentes parties d'un événement. 

Par exemple pour un tremblement de terre les sous-événements pourraient être :  
  - La secousse initiale
  - Les répliques sismiques
  - Les opérations de sauvetage
  - L'aide humanitaire
  - La reconstruction

<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=500 large=200/>
</p>

### __`Présentation de  la séquence :`__

#### __`Cohérence Générale`__

1. __Projection__
2. __Mesures de base (PageRank)__
3. __Embeddings__
4. __Échantillonnage__
5. __Clustering et détection de communautés__
6. __Analyse des communautés__
7. __Recommandations et similarité__
8. __Pathfinding et centralité__
9. __GraphSAGE pour des analyses avancées__  

__`Projection (gds.graph.project) :`__
 - Objectif : Charger le graphe en mémoire.
 - Analyse : Etape initiale de l'analyse avec GDS permettant de définir la topologie et les propriétés des nœuds/relations.

__`PageRank (gds.pageRank) :`__
 - Objectif : Calculer l'influence des nœuds.
 - Analyse : Très utile pour obtenir une première mesure de l'importance des nœuds. C'est une étape pour identifier les acteurs clés. 

__`Embeddings (gds.fastRP) :`__
 - Objectif : Générer des vecteurs d'embedding (représentations vectorielles denses des nœuds).
 - Analyse : Crucial pour de nombreuses tâches de Machine Learning sur graphes. Les embeddings capturent la structure du voisinage d'un nœud et ses propriétés. FastRP est rapide et efficace.

__`Échantillonnage (gds.graph.sample.cnarw) :`__
 - Objectif : Réduire la taille du graphe.
 - Analyse : cnarw (Common Neighbor Aware Random Walk) est une méthode d'échantillonnage qui tente de préserver la structure des communautés. Si le graphe est trop grand pour certains algorithmes ou pour la visualisation. 
 
__`La position de cette étape est intéressante :`__
 - Si nous échantillonnons après PageRank et Embeddings, nous avons des scores/vecteurs pour le graphe complet, puis nous travaillons sur un sous-ensemble.

Si l'on souhaite que les algorithmes suivants (comme k-means ou Leiden) s'exécutent plus rapidement, on pourrait échantillonner plus tôt, mais alors les scores/embeddings seraient calculés uniquement sur l'échantillon. La séquence actuelle suggère que les embeddings sont générés sur le graphe potentiellement plus grand.

__`Détection de Communautés (gds.Leiden.write) :`__
 - Objectif : Identifier les communautés naturelles basées sur la densité des connexions.
 - Analyse : Louvain est un algorithme populaire pour la détection de communautés structurelles. Le .write indique que les IDs des communautés sont écrits comme propriétés des nœuds, ce qui est utile pour des analyses ultérieures. Peut être exécuté sur le graphe complet ou l'échantillon.

__`Analyse des Communautés (gds.modularity.stats) :`__
 - Objectif : Étudier les propriétés (qualité) des communautés trouvées (probablement par Louvain).
 - Analyse : La modularité est une mesure clé pour évaluer la force de la division en communautés. Étape logique après la détection.

__`Recommandations (gds.alpha.ml) :`__
 - Objectif : Créer des systèmes de recommandation (probablement prédiction de liens ou de propriétés).
 - Analyse : Les pipelines ML de GDS (souvent en alpha ou beta) peuvent utiliser les embeddings (étape 3), les communautés (étape 6), ou d'autres caractéristiques pour faire des prédictions.

__`Similarité (gds.nodeSimilarity) :`__
 - Objectif : Identifier les nœuds similaires.
 - Analyse : Peut utiliser les embeddings (similarité cosinus sur les vecteurs FastRP) ou être basé sur la structure (Jaccard, Overlap). Utile pour trouver des "doublons" ou des nœuds ayant des rôles similaires.

__`Pathfinding (gds.alpha.shortestPath) :`__
 - Objectif : Trouver les chemins les plus courts.
 - Analyse : Fondamental pour comprendre la distance et la connectivité entre les nœuds.

__`Centrality (gds.alpha.closeness) :`__
 - Objectif : Identifier les nœuds centraux (ceux qui peuvent atteindre les autres rapidement).
 - Analyse : La centralité de proximité est une autre mesure d'importance, complémentaire à PageRank.

__`GraphSAGE (gds.beta.graphSage) :`__
 - Objectif : Effectuer des analyses de graphes (typiquement, générer des embeddings inductifs).
 - Analyse : GraphSAGE est un autre algorithme d'embedding puissant, souvent utilisé pour des graphes dynamiques ou pour généraliser à des nœuds non vus. Il pourrait être une alternative ou un complément à FastRP. Son placement en fin de liste suggère une exploration plus avancée.

#### __`1.1 Connexion`__

In [22]:
import os
import pandas as pd
import numpy as np
from neo4j import GraphDatabase
from graphdatascience import GraphDataScience

In [23]:
def get_gdsConnection():
    # Get Neo4j DB URI, credentials and name from environment if applicable
    NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
    NEO4J_AUTH = ("neo4j", "gbenitah")
    NEO4J_DB = os.environ.get("NEO4J_DB", "neo4j")
    if os.environ.get("NEO4J_USER") and os.environ.get("NEO4J_PASSWORD"):
        NEO4J_AUTH = (
            os.environ.get("NEO4J_USER"),
            os.environ.get("NEO4J_PASSWORD"),
            )
    gds = GraphDataScience(NEO4J_URI, auth=NEO4J_AUTH, database=NEO4J_DB)
    print(gds.server_version())
    return gds

#### __`1.2 Outils de libération de la mémoire efface les graphes projetés`__

In [24]:
# Permet de décharger les projections de la mémoire et de Neo4j
query_drop = f"""
    CALL gds.graph.exists($graphName)
         YIELD exists
         WITH exists
    WHERE exists = true
    CALL gds.graph.drop($graphName, false)
         YIELD graphName
    RETURN graphName;
    """

query_inspect_graph = """
    CALL gds.graph.list()
    YIELD graphName, schema
    RETURN distinct graphName
    """
#gds.run_cypher(query_inspect_graph)

def drop_wholeGraph(gds): 
    result = gds.run_cypher(query_inspect_graph)
    for index, row in result.iterrows():
        gds.run_cypher(query_drop, params={'graphName': row['graphName']})
        
gds=get_gdsConnection()        

2.13.4


#### __`1.3 Outils de réinitialisation des attributs de la base Neo4J`__

In [25]:
# Retire tous les attribus qui ont été ajoutés aux noeuds dans l'environnement Neo4j
def clean_database_parameter(gds, lisparam=[]):
    query_removeAttribute= ("""
        MATCH (n)
        WITH n, [k IN keys(n) WHERE ANY(param IN $parameter WHERE k CONTAINS param) | k] AS propertyKeys
        FOREACH (i IN propertyKeys | REMOVE n[i])
        RETURN n
        """)
    gds.run_cypher(query_removeAttribute, params={'parameter': lisparam})

clean_database_parameter(gds,lisparam= [ 'topicId','eventTypeId', 'eigenvector', 'betweeness', 'rank',
                                        'degree', 'localClusteringCoefficient', 'intermediateCommunities', 
                                        'pageRank', 'articulationPoint','community'])

In [26]:
# Retire tous les attribus qui ont été ajoutés aux noeuds sur le graphe projeté
def clean_projectionGraph_Parameter(gds, graphName, parameter):
        query_drop = """    
        CALL gds.graph.nodeProperties.drop($graphName, $parameter)
        YIELD propertiesRemoved, graphName
        RETURN propertiesRemoved, graphName
        """
        graph = gds.graph.get(graphName)
        node_properties_by_label = graph.node_properties()
        for label, properties_dict in node_properties_by_label.items():
            if parameter in properties_dict:
               gds.run_cypher(query_drop, params={'graphName': graphName, 'parameter': parameter})
               print(f"Propriété '{parameter}' supprimée du graphe '{graphName}'.")
               break

In [27]:
# retourne la liste des graphe en mémoire
def list_graphs(gds):
    query_listPjtBase="""CALL gds.graph.list() YIELD graphName, nodeCount, relationshipCount, schema RETURN distinct graphName, nodeCount, relationshipCount, schema"""
    return gds.run_cypher(query_listPjtBase)

#### __`1.4 Création de "topicId"`__

Le filtrage sur les graphes projetés ne peut s'effectuer que sur des attributs numériques. Pour permettre le filtrage sur les topics il est nécessaire de céer un attribut numérique qui match avec l'attribut topic et ce pour le label Event et Tweet.

In [28]:
# On récupère les deux derniers caractères numériques de trecisid pour en faire un ID en correspodance avec l'id de Event 
# l'objectif est de peremttre d'effectuer sur les graphe projeté des restrictions sur les topics à partir de Event et de Tweet.

query_set_topicid_corrected = """
    MATCH (e:Event)
    SET e.topicId = toIntegerOrNull(substring(e.trecisid, size(e.trecisid) - 2, 2))
    WITH count(e) AS events_processed

    MATCH (t:Tweet)
    SET t.topicId = toIntegerOrNull(substring(t.topic, size(t.topic) - 2, 2))

    RETURN events_processed, count(t) AS tweets_processed
    """

try:
    update_result = gds.run_cypher(query_set_topicid_corrected)
    # Affiche le nombre d'events et tweets potentiellement mis à jour
except Exception as e:
    print(f"Erreur lors de la mise à jour des topicId: {e}")
    # Gérer l'erreur

query_dico_topicId = """
    MATCH (e:Event)
    WHERE e.topicId IS NOT NULL
    RETURN e.topicId AS key, e.id AS value
    """
dico_topicId = gds.run_cypher(query_dico_topicId)
# on crée un dictionnaire pour récupérer les valeurs littérales à partir des ident
dico_topicId = dict(zip(dico_topicId['key'], dico_topicId['value']))  

#### __`1.5 Création de "eventTypeId"`__

Le filtrage sur les graphes projetés ne peut s'effectuer que sur des attributs numériques. Pour permettre le filtrage sur les EventType il est nécessaire de céer un attribut numérique qui match avec l'attribut Eventype pour le label Event.

In [29]:
# Il est nécessaire de créer un attribut numérique qui soit en corrrespondance avec les "Event" afin de permettre 
# l'utilisation des fonctionnalités GDS pour de création de sous-graphe par "Event". 
query= """
    MATCH (n:Event)
    SET n.eventTypeId = CASE n.eventType
        WHEN 'bombing' THEN toInteger(1)
        WHEN 'shooting' THEN toInteger(2)
        WHEN 'earthquake' THEN toInteger(3)
        WHEN 'typhoon' THEN toInteger(4)
        WHEN 'flood' THEN toInteger(5)
        WHEN 'wildfire' THEN toInteger(6)
        ELSE 0 END;
    """

out=gds.run_cypher(query)

In [30]:
# vérification de la création de l'atrribut numérique relatif aux différents "Event"
query_checkEventId = """MATCH (n:Event) RETURN n.id, n.topicId,n.eventType, n.eventTypeId"""
gds.run_cypher(query_checkEventId)

,n.id,n.topicId,n.eventType,n.eventTypeId
0,fireColorado2012,1,wildfire,6
1,costaRicaEarthquake2012,2,earthquake,3
2,floodColorado2013,3,flood,5
3,typhoonPablo2012,4,typhoon,4
4,laAirportShooting2013,5,shooting,2
5,westTexasExplosion2013,6,bombing,1
6,guatemalaEarthquake2012,7,earthquake,3
7,italyEarthquakes2012,8,earthquake,3
8,philipinnesFloods2012,9,flood,5
9,albertaFloods2013,10,flood,5


In [31]:
# vérification de la création de l'atrribut numérique relatif aux différents "Event"
query_checkEventId = """MATCH (n:Tweet) RETURN n.id, n.topicId, n.topic"""
df_alt_replace = gds.run_cypher(query_checkEventId)
df_alt_replace

,n.id,n.topicId,n.topic
0,212364712506163201,1,TRECIS-CTIT-H-001
1,212365530391252993,1,TRECIS-CTIT-H-001
2,212373109477617664,1,TRECIS-CTIT-H-001
3,212377068904787969,1,TRECIS-CTIT-H-001
4,212384295719931904,1,TRECIS-CTIT-H-001
...,...,...,...
55981,212311709082337281,1,TRECIS-CTIT-H-001
55982,212311994286620672,1,TRECIS-CTIT-H-001
55983,212315735609966592,1,TRECIS-CTIT-H-001
55984,212319443383103488,1,TRECIS-CTIT-H-001


__`Avant d'appliquer des algorithmes complexes, explorons certaines caractéristiques fondamentales`__

  __`1) Base de l’embedding`__:      
  - Les techniques d’embedding basées sur "PageRank", reposent sur la génération préalable de vecteurs PageRank.      
  - Ces vecteurs capturent l’importance structurelle ou locale des nœuds dans le graphe, qui est ensuite transformée en une représentation compacte via des méthodes comme la décomposition en valeurs singulières (SVD) ou d’autres algorithmes d’embedding.

  __`2) Processus typique`__ :     
   - D’abord, les vecteurs PageRank (ou Personalized PageRank) sont calculés pour les nœuds du graphe.     
   - Ensuite, ces vecteurs sont transformés, en appliquant soit une opération logarithmique ou une pondération spécifique.         
   - Puis avant on passe à l'étape d’embedding comme le SVD pour produire les représentations finales.          

### __`2 Projection chargement des graphes en mémoire`__  

La première étape consiste à projeter le graphe principal en mémoire pour accélérer les calculs et les analyses.        
Cette opération utilise les fonctions de GDS de graphe projeté qui n'autorise pas l'utilisation d'attribut autre que numériques permettre le filtrage des noeuds du graphe par type d'event tels qu'ils apparaissent dans notre exemple. 

Par principe de précaution, il est recommandé de toujours projeter le graphe avant d'effectuer des calculs ou des analyses.         
La projection du graphe est une étape cruciale pour l'analyse de graphes volumineux, car elle permet de créer une vue en mémoire optimisée du graphe qui accélère les calculs et les requêtes.

In [32]:
# Efface de l'environnement mémoire les graphes projetés produits
drop_wholeGraph(gds)

<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=500 large=200/>
</p>

In [33]:
query_create_main_graph = """
    CALL gds.graph.project(
        $graphName,
        {
          Event:        { properties: $properties[0] },
          User:         { properties: $properties[1] },
          Tweet:        { properties: $properties[2] },
          Hashtag:      { properties: $properties[3] },
          PostCategory: { properties: $properties[4] }
        },
        {
          POSTED:      { orientation: $orientation[0] },
          MENTIONS:    { orientation: $orientation[1] },
          REPLIED_TO:  { orientation: $orientation[2] },
          TALKS_ABOUT: { orientation: $orientation[3] },
          REPLY_TO:    { orientation: $orientation[4] },
          IS_ABOUT:    { orientation: $orientation[5] },
          HAS_HASHTAG: { orientation: $orientation[6] },
          HAS_CATEGORY:{ orientation: $orientation[7] },
          RETWEETS:    { orientation: $orientation[8] },
          RETWEETED:   { orientation: $orientation[9] }
        }   
    )
       YIELD
        graphName,
        nodeCount,
        relationshipCount,
        nodeProjection,         
        relationshipProjection, 
        projectMillis           
      RETURN graphName, nodeCount, relationshipCount, nodeProjection, relationshipProjection, projectMillis;
"""

### __`3 PageRanking`__  

L'algorithme PageRank permet de mesurer la pertinence de chaque nœud d'un graphe, qui dépend du nombre de relations entrantes provenant des autres nœuds et de l'importance des nœuds sources. 


##### __`3.1 Centralité basée sur le pageRank`__ 

:::  

__Le PageRank mesure l'importance du nœud fonction du nombre de relations entrantes et de l'importance des nœuds sources correspondants.     
L'hypothèse sous-jacente est qu'une page n'a d'importance que si elle est liée à des pages.__ 

$PR(A)=(1-d)+d\large \frac{PR(T_i)}{C(T_i)}+...+d\large \frac{PR(T_n)}{C(T_n)}$
 - nous supposons qu'une page A possède des pages $T_1$ à $T_n$ qui pointent vers elle.
 - $d$ est un facteur d'amortissement réglable entre 0 (inclus) et 1 (exclu). Il est généralement fixé à 0,85.
 - $C(A)$ est défini comme le nombre de liens sortant de la page $A$.

:::



In [34]:
# Utiliser uniquement les labels existants
query_pageRank_mutate = """
    CALL gds.pageRank.mutate(
        $graphName,
      { 
        mutateProperty: $nodeProperties,
        maxIterations: 20,
        dampingFactor: 0.85,
        nodeLabels: $nodeLabels, 
        relationshipTypes: $relationshipTypes 
      }
    )
   """

query_pageRank_stream=""" 
    CALL gds.graph.nodeProperty.stream(
        $graphName, 
        $nodeProperties,
        $nodeLabels, 
        $relationshipTypes 
    )
    YIELD nodeId, propertyValue AS score 
    WITH gds.util.asNode(nodeId) AS userNode, score
    RETURN userNode.name AS influentialUser, score
    ORDER BY score DESC
    """

::: 


Il y a quelques points à prendre en considération lors de l'utilisation de l'algorithme PageRank :

 - S'il n'existe pas de relation entre l'intérieur et l'extérieur d'un groupe de pages, celui-ci est considéré comme un piège à araignées.    
 - Une chute de rang peut se produire lorsqu'un réseau de pages forme un cycle infini.    
 - Les impasses se produisent lorsque les pages n'ont aucune relation sortante.    

Modifier le facteur d'amortissement peut contribuer à toutes les considérations ci-dessus.     
Il peut être interprété comme la probabilité qu'un internaute accède parfois à une page aléatoire et évite ainsi de se retrouver bloqué dans des éviers.

:::

### __`4 Echantillonnage & Réduction du graphe`__  

L'algorithme CNARW, (Common Neighbour Aware Random Walk) est conçu pour créer des échantillons équilibrés de grands graphes tout en préservant leurs propriétés structurelles. 

Contrairement aux marches aléatoires simples qui se retrouvent souvent piégées dans des zones hautement groupées d'un graphe, CNARW navigue intelligemment dans le graphe en :   

 1. Considérant les voisins communs entre le nœud actuel et les nœuds potentiels suivants
 2. Priorisant les chemins qui mènent vers des régions inexplorées du graphe
 3. Réduisant la probabilité de revisiter des nœuds déjà visités

Cette approche est particulièrement précieuse pour les sous-graphes basés sur des événements car elle aide à maintenir la nature représentative de chaque communauté d'événements tout en réduisant le graphe à une taille plus gérable.     

Dans la plupart des approches de clustering basées sur des graphes, il est recommandé de réaliser la réduction du graphe avant l’identification des clusters afin de bénéficier d’une meilleure performance et d’une plus grande robustesse dans l’analyse structurelle du réseau.

In [35]:
query_random_walk = """
    CALL gds.graph.sample.rwr(
      $sampled_graph_name,
      $subgraph_name,
      { 
        samplingRatio: $sampling_ratio,
        nodeLabelStratification: true,
        nodeProperties: $nodeProperties 
      }
    )
    YIELD nodeCount, relationshipCount;
    """

query_random_walk_estimate="""
  CALL gds.graph.sample.cnarw.estimate(
    $subgraph_name,
  {
    samplingRatio: 0.25
  })
  YIELD requiredMemory
  RETURN requiredMemory
  """   
#    startNodes: $start_node_ids 
#    walkLength: 4,
#    iterations: 10, 

### __`5 Embeddings`__

Les algorithmes d'intégration de nœuds calculent des représentations vectorielles de faible dimension des nœuds d'un graphe. Ces vecteurs, également appelés intégrations, peuvent être utilisés pour l'apprentissage automatique. La bibliothèque Neo4j Graph Data Science contient l'algorithme d'intégration de nœuds suivant : 

 - __La projection aléatoire rapide, ou FastRP__, est un algorithme d'intégration de nœuds appartenant à la famille des algorithmes de projection aléatoire. 
 
 - __Ces algorithmes reposent théoriquement sur le lemme de Johnsson-Lindenstrauss__ selon lequel il est possible de projeter n vecteurs de dimension arbitraire en O(log(n)) dimensions tout en préservant approximativement les distances deux à deux entre les points. 

En fait, une projection linéaire choisie aléatoirement satisfait cette propriété.

Nous utilisons l'algorithme FastRP pour la génération de l'embedding des noeuds, il fournit une représentation topographique du graphe.      
Empiriquement nous avons choisi le paramètre embeddingDimension à 8 au regard de la densité du graphe.

Les embeddings FastRP s’appliquent à tous les nœuds présents dans le graphe projeté, indépendamment de leurs labels.     
Si la projection contient différents types de nœuds, l’algorithme générera des embeddings pour chacun d’eux, contrairement à ce que l’on pourrait penser, FastRP ne traite pas séparément les différents types de nœuds. FastRP traite le graphe comme homogène.

Les algorithmes d’embedding comme FastRP génèrent des vecteurs qui capturent les relations structurelles entre tous les types de nœuds dans un espace vectoriel commun.

In [36]:
query_mutate_embeddings = """
CALL gds.fastRP.mutate(
  $graphName,
  {
    embeddingDimension : 128,   
    mutateProperty: $mutateProperty,
    iterationWeights : [0.8, 0.2, 0.3 ],
    normalizationStrength : 0.01,
    propertyRatio : 0,
    randomSeed : 42,
    nodeLabels : $nodeLabels,
    relationshipTypes: $relationshipTypes   
  }
)
YIELD nodePropertiesWritten, computeMillis;
"""

In [37]:
query_stream_embeddings=  f""" 
CALL gds.fastRP.stream(
  $graphName,
  {{
    embeddingDimension: 128,   
    iterationWeights: [1,1,1,1,1 ],
    normalizationStrength: 0.01,
    propertyRatio: 0,
    randomSeed: 42,
    nodeLabels: ['User','Event', 'Tweet', 'Hashtag', 'PostCategory' ] ,
    relationshipTypes: ['HAS_HASHTAG', 'MENTIONS', 'POSTED', 'REPLIED_TO', 'REPLY_TO', 'RETWEETS']   
  }}
)
YIELD nodeId, embedding
RETURN embedding,  
  CASE 
    WHEN gds.util.asNode(nodeId):Hashtag THEN 'Event'
    WHEN gds.util.asNode(nodeId):User THEN 'User'
    WHEN gds.util.asNode(nodeId):Tweet THEN 'Tweet'
    WHEN gds.util.asNode(nodeId):Hashtag THEN 'Hashtag'
    WHEN gds.util.asNode(nodeId):PostCategory THEN 'PostCategory'
  END AS Label,
    COALESCE(gds.util.asNode(nodeId).eventType,gds.util.asNode(nodeId).name) AS Event,
    gds.util.asNode(nodeId).id as id,
    gds.util.asNode(nodeId).eventType AS eventType,
    COALESCE(gds.util.asNode(nodeId).trecisid,gds.util.asNode(nodeId).topic,gds.util.asNode(nodeId).name ) AS Topic
ORDER BY Label
""" 

#df = pd.DataFrame() 
#result = gds.run_cypher(query_inspect_graph)
#for index, row in result.iterrows():
#    print(row)
#    data=gds.run_cypher(query_stream_embeddings, params={'graphName': row['graphName']})
#    df = pd.concat([df, data], ignore_index=True)
#df


### __`6 Analyse Communautaire`__

#### __` 6.1 Analyse communautaire, l'algorithme de Leiden`__ 

:::

L'algorithme de Leiden est un algorithme de détection de communautés dans les grands réseaux.    
Il sépare les nœuds en communautés disjointes afin de maximiser le `score de modularité` de chaque communauté.   
  
La modularité quantifie la qualité de l'affectation des nœuds aux communautés, c'est-à-dire la densité de connexion des nœuds d'une communauté par rapport à leur niveau de connectivité dans un réseau aléatoire.

L'algorithme de Leiden  modifie l' algorithme de Louvain pour corriger certaines de ses lacunes, notamment lorsque certaines communautés trouvées par Louvain ne sont pas bien connectées.    

:::

##### __` 6.1.1 Application sur plusieurs types de nœuds (Graphe Hétérogène)`__ 

:::

Si l'on souhaite trouver des communautés basées sur les interactions entre différents types d'entités. Par exemple, trouver des groupes de User et de Tweet qui sont fortement liés (des utilisateurs qui interagissent beaucoup avec un certain ensemble de tweets, et ces tweets sont postés par ces mêmes utilisateurs ou d'autres utilisateurs du groupe).

:::

In [38]:
# leiden UNDIRECTED graph heteregeneous nodes 
# Création des clusters avec Leiden
query_leiden_mutate = """
    CALL gds.leiden.mutate(
        $graphName, 
        {
            mutateProperty: $intermediateCommunities,
            randomSeed: 19,
            includeIntermediateCommunities: true,
            concurrency: 1
        }
        )
    YIELD communityCount, modularity, modularities
    """ 

query_leiden_write = """
    CALL gds.leiden.write(
        $graphName,  
        {
            writeProperty: $intermediateCommunities,
            randomSeed: 19,
            includeIntermediateCommunities: true,
            concurrency: 1
        }
        )
    YIELD communityCount, modularity, modularities
    """

query_leiden_stream = """
CALL gds.leiden.stream(
        $graphName,  
        {
            randomSeed: 19
        })
YIELD nodeId, communityId
RETURN gds.util.asNode(nodeId).name AS name, communityId
ORDER BY name ASC
"""

:::

__`La modularité`__  
  
 - La modularité est une mesure de la structure des réseaux qui évalue la force de la division d'un réseau en modules ou communautés.    
 - Une valeur de modularité élevée indique une structure de communauté forte, avec des connexions denses entre les nœuds au sein des modules et des connexions éparses entre les nœuds de différents modules. 
 
Dans le cas de gds.leiden.write, la modularité retournée est un nombre flottant qui quantifie la qualité de la partition en communautés trouvée par l'algorithme Leiden. Plus la valeur est proche de 1, meilleure est la structure de communauté détectée.

:::

#### __`6.2 Analyse communautaire, l'algorithme de propagation d'étiquettes (LP)`__ 

:::

LPA est un algorithme rapide permettant de trouver des communautés dans un graphe.     
Il détecte ces communautés en se basant uniquement sur la structure du réseau.     
LPA fonctionne en propageant des étiquettes sur tout le réseau et en formant des communautés basées sur ce processus de propagation d'étiquettes.

L'algorithme fonctionne comme suit :

 - Chaque nœud est initialisé avec une étiquette de communauté unique (un identifiant).
 - Ces étiquettes se propagent à travers le réseau.

À chaque itération de propagation, chaque nœud met à jour son étiquette pour correspondre à celle à laquelle appartient le plus grand nombre de ses voisins. Les liens sont rompus de manière arbitraire, mais déterministe.

 - LPA atteint la convergence lorsque chaque nœud possède l'étiquette majoritaire de ses voisins.
 - LPA s'arrête si la convergence ou le nombre maximal d'itérations défini par l'utilisateur est atteint.

 :::

In [39]:
# labelPropagation Directed & Undirected graph, Heterogenous nodes
# Création des clusters avec labelPropagation
query_labelPropagation_mutate = """
    CALL gds.labelPropagation.mutate(
        $graphName, 
        { 
            mutateProperty: $community
        }
        )
    YIELD nodePropertiesWritten, communityCount, ranIterations, didConverge
    RETURN nodePropertiesWritten, communityCount, ranIterations, didConverge
    """

:::

__`labelPropagation comportement Implicite : Non Orienté (Undirected)`__

L'objectif est d'utiliser la structure du réseau pour trouver les communautés   

 - Chaque nœud adopte l'étiquette majoritaire parmi ses voisins.
 - Cette notion de "voisinage" pour la détection de communauté est intrinsèquement non orientée. 
 - L'algorithme cherche des groupes denses où les nœuds sont connectés, peu importe la direction initiale du lien.
 - L'algorithme est intrinsèquement non orienté, il se concentre sur la structure globale du réseau plutôt que sur la direction des liens. 
 - Il est à noter que l'algorithme peut être appliqué à des graphes orientés, mais le comportement sera similaire à celui d'un graphe non orienté.

 :::
 

:::

### __`6.3 Analyse communautaire, Coefficient de clustering local`__

__`Le coefficient de clustering local évalue la densité de connexion locale entre les voisins immédiats d'un nœud`__

Le coefficient de clustering local représente la proportion des liens existants entre les voisins d'un nœud par rapport au nombre maximum de liens possibles entre eux. L'algorithme permet de mesurer la densité de connexion locale $\Large C_n = \frac{T_n}{d_n(d_n - 1)}$
- $T_{n}$ est le nombre de triangles dont le nœud $n$ fait partie.
- $d_{n}$ est le degré du nœud $n$. 
- $d_{n}(d_{n} - 1)$ est le nombre maximum de liens possibles entre les voisins du nœud $n$.
- $C_{n}$ est le coefficient de clustering local du nœud $n$.
- $C_{n}$ est calculé pour chaque nœud du graphe, et il peut être utilisé pour identifier les nœuds qui sont fortement connectés à leurs voisins.   

:::

In [40]:
# localClusteringCoefficient Undirected graph, Heterogenous nodes
localClstrCoef_query_mutate = """
    CALL gds.localClusteringCoefficient.mutate(
    $graphName, 
    {
        mutateProperty: $localClusteringCoefficient
    }
    )
    YIELD averageClusteringCoefficient, nodeCount
    """    

:::

__Interprétation des Valeurs :__

Le coefficient de clustering local est une valeur comprise entre 0 et 1

 - __Valeur proche de 1__ : Indique que la plupart (ou tous) les voisins du nœud sont connectés entre eux. Le voisinage du nœud ressemble fortement à une clique (un groupe où tout le monde est connecté à tout le monde). Un nœud avec un LCC de 1.0 a tous ses voisins directement connectés les uns aux autres.

 - __Valeur proche de 0__ : Indique que les voisins du nœud sont peu ou pas connectés entre eux. Le nœud peut agir comme un "pont" entre différents groupes ou individus qui ne se connaissent pas directement. Un LCC de 0.0 signifie qu'aucun des voisins du nœud n'est connecté à un autre voisin.

:::

### __`7 Identification de points critiques`__ 

### __`7.1 Point d'articulation`__

::: 

__`Un point d'articulation est un nœud dont la suppression augmente le nombre de composants connectés dans le graphe.`__
 - Il est essentiel pour maintenir la connectivité du réseau.
 - La suppression d'un point d'articulation peut fragmenter le réseau en plusieurs sous-graphes.
 - Les points d'articulation sont souvent des nœuds centraux ou des hubs.
 - Ils jouent un rôle clé dans la diffusion d'information et la résilience du réseau.
 - L'algorithme de détection des points d'articulation est basé sur la recherche en profondeur (DFS).
 - Il identifie les nœuds critiques en analysant les connexions entre les nœuds.
 - Les points d'articulation sont souvent utilisés pour optimiser la structure du réseau.
 - Ils peuvent être utilisés pour cibler des interventions stratégiques.

:::





:::

__`Implications pratiques`__  

__Vulnérabilité structurelle__ : Ces points d'articulation constituent des points de défaillance uniques dont la suppression fragmenterait davantage le réseau
__Contrôle informationnel__ : Ces nœuds exercent un contrôle disproportionné sur les flux d'information entre différentes parties du réseau     
__Cibles d'intervention__ : Pour une intervention stratégique, ces nœuds offriraient le meilleur rapport impact/effort.
 
 :::

Étant donné un graphe, un point d'articulation est un nœud dont la suppression augmente le nombre de composantes connexes du graphe. La bibliothèque GDS Neo4j fournit un algorithme séquentiel linéaire efficace pour calculer tous les points d'articulation d'un graphe.

In [41]:
# ArticulationPoint Directed & Undirected graph, Heterogenous nodes
# MATCH (n { articulationPoint: 1 }) RETURN n.name AS name ORDER BY name ASC
articulationPoints_query_mutate = """
    CALL gds.articulationPoints.mutate(
    $graphName,{  mutateProperty: $articulationPoint})
    YIELD articulationPointCount
    """

:::

## __`7.1 Appliquer des Algorithmes Avancés`__  

Nous pouvons à présent appliquer certains algorithmes avancés sur les graphes échantillonnés sans risque de conflit.   
Calcule des mesures de centralité telles que la centralité de degré, d'intermédiarité, et basées sur les valeurs propres (eigenvector, PageRank) pour identifier les nœuds clés.

:::


### __`7.1.1 Mesures de centralité`__

:::

#### __`7.1.1.1 Centralité de degré`__  

L'algorithme de centralité de degré permet de trouver les nœuds les plus fréquents dans un graphe.       
Degré mesure le nombre de relations entrantes et/ou sortantes d'un nœud, qui peut être défini par l'orientation d'une projection de relations. 

__`Définition`__ : La centralité de degré compte les relations.

 - `Défaut GDS (gds.degree.*)` : L'orientation par défaut est NATURAL.
 - `Comportement de NATURAL (Défaut)` : Calcule le degré sortant (out-degree).
 - `Pour calculer le degré entrant (In-degree)` : Utilisez orientation: 'REVERSE'.
 - `Pour calculer le degré total (In + Out)` : Utilisez orientation: 'UNDIRECTED'

:::


In [42]:
# Calcul des degrés Directed & NATURAL graph, Heterogenous nodes
degree_query_mutate = """
    CALL gds.degree.mutate(
    $graphName, 
    { 
        mutateProperty: $degree 
    }
    )
    YIELD centralityDistribution, nodePropertiesWritten
    RETURN centralityDistribution.min AS minimumScore, centralityDistribution.mean AS meanScore, nodePropertiesWritten
    """ 

:::

#####  __` 7.1.1.1.1 Mesure de l'Activité ou de l'Influence Émise`__ :    

Le score de centralité de degré ne reflète plus simplement le nombre total de connexions, mais spécifiquement le nombre de connexions initiées par ou sortant de ce nœud. 

Il mesure donc l'activité d'un nœud en tant qu'émetteur, source ou initiateur de relations.

 - `Pour un nœud User, un degré sortant élevé signifierait que l'utilisateur est très actif` : il poste (POSTED), répond (REPLY_TO), mentionne (MENTIONS), retweete (RETWEETS) ou parle d'événements (TALKS_ABOUT) fréquemment. Il mesure l'activité sortante de l'utilisateur.

:::

:::

#### __`7.1.1.2 Centralité d'intermédiarité`__ 

La centralité d'intermédiarité permet de détecter l'influence d'un nœud sur le flux d'information d'un graphe.     
Elle est souvent utilisée pour identifier les nœuds servant de passerelle entre les différentes parties d'un graphe.

`Elle permet de détecter l'influence d'un nœud sur le flux d'information d'un graphe`.    
`Elle est souvent utilisée pour identifier les nœuds servant de passerelle entre les différentes parties d'un graphe`.    

L'implémentation GDS est basée sur l'algorithme approximatif de Brandes pour les graphes non pondérés.    

 - L'algorithme calcule les chemins les plus courts entre toutes les paires de nœuds d'un graphe.       
 - Chaque nœud reçoit un score basé sur le nombre de chemins les plus courts qui le traversent.         
 - Les nœuds qui se trouvent le plus souvent sur les chemins les plus courts entre d'autres nœuds auront des scores plus élevés.

Cette métrique mets en lumière les nœuds qui servent de liaison entre plusieurs clusters et ainsi identifient les utilisateurs jouant un rôle de pont inter-évènements.

:::

In [43]:
# Betweenness centralités intermédiaires Directed & NATURAL graph, Heterogenous nodes 
betweenness_query_mutate = """
    CALL gds.betweenness.mutate(
    $graphName, 
        { 
        mutateProperty: $betweenness 
        }
    )
    YIELD centralityDistribution, nodePropertiesWritten
    RETURN centralityDistribution.min AS minimumScore, centralityDistribution.mean AS meanScore, nodePropertiesWritten
    """     

::: 
#### __`7.1.1.3 Betweeness et le graphe Orienté (Directed)`__

__`Fonctionnement`__ :   

 - L'algorithme calcule les chemins les plus courts en respectant la direction des relations. 
 - Un chemin de A à C n'est valide que s'il suit les flèches (A -> B -> C). 
 - La Betweeness Centrality d'un nœud est basée sur le nombre de ces chemins orientés les plus courts qui passent par lui.

 - `Avantage` : 
     - C'est le choix pertinent lorsque le flux (d'information, de ressources, etc.) a une direction spécifique et que vous voulez identifier les nœuds qui contrôlent ou facilitent ce flux directionnel.

     - __Dans notre modèle, cela permettrait d'identifier les User ou Tweet qui sont cruciaux pour connecter des parties du réseau en suivant les interactions spécifiques (qui MENTIONS qui, qui REPLIED_TO quoi, etc.).__

 - `Contrainte/Implication` : Il ne mesure que l'intermédiarité dans le contexte du flux directionnel. Un nœud peut être un pont structurel important si l'on ignore la direction, mais avoir une faible BC dirigée s'il ne se trouve pas sur de nombreux chemins orientés les plus courts.

:::

:::

__`Hétérogénéité des Nœuds`__

La Centralité d'Intermédiarité dans GDS supporte les graphes avec des nœuds hétérogènes. Comme pour les autres algorithmes de centralité, le calcul lui-même traite tous les nœuds de manière structurelle similaire, en se basant sur leur position dans le réseau.

`L'interprétation du score de BC dépend fortement du type de nœud` :       
 - Un User avec une BC élevée est un intermédiaire important entre d'autres utilisateurs ou groupes d'utilisateurs.    
 - Un Tweet avec une BC élevée est un point de passage crucial pour les conversations, reliant potentiellement différents sujets ou utilisateurs.   
 - Un Hashtag ou Event avec une BC élevée connecte des discussions ou des utilisateurs qui autrement ne seraient pas liés directement.


::: 

#### __`7.1.1.4 Centralité des vecteurs propres`__

La centralité eigenvector repose sur le principe qu'un nœud est influent s'il est connecté à d'autres nœuds influents.
Contrairement à la simple centralité de degré, elle capture les dynamiques de pouvoir indirectes dans un réseau.
    
Un score de vecteur propre élevé signifie qu'un nœud est connecté à de nombreux nœuds ayant eux-mêmes des scores élevés. Les points à prendre en compte lors de l'utilisation de l'algorithme de centralité des vecteurs propres :

 - Les scores de centralité des nœuds sans relations entrantes convergent vers 0.
 - Du à l'absence de normalisation du degré, les nœuds de degré élevé ont une forte influence sur le score des voisins.

Pour un graphe d'adjacence $A$, le score eigenvector $x_i$ d'un nœud $i$ satisfait : $ \lambda x_i = \sum_{j\in N(i)} A_{ij} x_j$   
où :
 - $\lambda$ = plus grande valeur propre (dominante)
 - $N(i)$ = voisins directs de $i$

::: 

In [44]:
# Eigenvector vecteurs propres Directed & NATURAL graph, Heterogenous nodes
eigenvector_query_mutate="""
    CALL gds.eigenvector.mutate(
    $graphName, 
    { 
        maxIterations: 20, 
        mutateProperty: $eigenvector
     }
    )
    YIELD nodePropertiesWritten, ranIterations
    """

:::

#### __`7.1.1.5 EigenVector et le graphe orienté (Directed)`__
    
__Fonctionnement :__ Dans un graphe orienté, l'algorithme prend en compte la direction des relations. Le calcul standard de la Centralité de Vecteur Propre, tel qu'implémenté dans GDS, base le score d'un nœud sur les scores de ses voisins entrants (les nœuds qui pointent vers lui).

   - __`Avantage`__ : C'est le choix le plus pertinent lorsque les relations représentent un flux d'influence ou de dépendance directionnel. 
   
     - Par exemple, dans le modèle fourni, l'influence d'un Tweet (son score EC) serait déterminée par l'importance des User qui l'ont RETWEETED ou qui lui ont REPLIED_TO (relations entrantes vers Tweet), ce qui correspond bien à l'idée d'influence sur les réseaux sociaux. Un User serait influent s'il reçoit des MENTIONS ou des REPLIES de la part d'autres utilisateurs importants.  

   - __`Contrainte/Implication`__ : L'influence ne se propage que dans le sens des relations. Les relations sortantes d'un nœud (par exemple, un User qui POSTED un Tweet) contribuent au score du nœud de destination (Tweet), mais pas directement au score du nœud source (User).

:::

:::

### __`7.1.2 Ecriture des variables sur Neo4j`__

|Élément	|Comportement	|Solution|
|:-:        |---                |---     |
|Mode Write	|Écrit dans Neo4j uniquement |Ne modifie pas la projection GDS|
|Mode Mutate	|  - |Stocke dans la projection GDS|
|gds.graph.writeNodeProperties| -	|Projection → Neo4j|


:::

In [45]:
# Ecrit des propriétés de nœuds depuis un graphe projeté en mémoire (dans le catalogue GDS) vers la base de données Neo4j
neo4j_writeNodesProperties="""
CALL gds.graph.writeNodeProperties(
    $graphName,
    $propertiesList,
    $nodeLabels
)
"""

<p style="text-align: center">
<img src="images/model.png" alt="Olympics Games" width=500 large=200/>
</p>

## __`7.2 Affichage des graphes projetés`__

In [46]:
import matplotlib.pyplot as plt
import matplotlib.colors
from neo4j_viz.gds import from_gds

def display_intermediate(gds,graphName,nodeProperties):
    # Créer une fonction pour colorer les communautés par niveau
    graph_obj = gds.graph.get(graphName)
    nodes = gds.graph.nodeProperties.stream(graph_obj,[f'{nodeProperties}'])
    clustering_values = nodes['propertyValue'].dropna()

    vmin = min(clustering_values)
    vmax = max(clustering_values)
    normalize = plt.Normalize(vmin=vmin, vmax=vmax)
    colormap = plt.cm.plasma  # Palette continue sans limite de couleurs

    node_id_to_color_map = {
        row['nodeId']: matplotlib.colors.rgb2hex(colormap(normalize(row['propertyValue'])))
        if pd.notna(row['propertyValue']) else 'red' # Couleur grise pour NaN
        for index, row in nodes.iterrows()
            }

    VG = from_gds(
            gds,
            graph_obj,
            size_property=f'{nodeProperties}',
            node_radius_min_max = (20, 600)
        )

    VG.color_nodes('id', node_id_to_color_map)

    return VG